# JobSpy local test

Run and test [JobSpy](https://github.com/speedyapply/JobSpy) (job scraper for LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter, etc.).  
**Setup:** `uv sync` in this folder, then run cells. Kernel: use the `.venv` from this project.

*Structured for future Marimo conversion: small result sets, minimal globals.*

## 1. Check environment

In [ ]:
import sys
import jobspy

print(f"Python: {sys.version}")
print(f"jobspy: {getattr(jobspy, '__version__', 'unknown')}")
print("Import OK.")

## 2. Search UI — set options and run

Pick sites, search term, location (preset or **Other** for custom), **Remote** filter, and other params.  
*Note: Indeed allows only one of: hours_old, (job_type + is_remote), or easy_apply. LinkedIn: only one of hours_old or easy_apply.*

In [ ]:
import csv
from IPython.display import display as ipy_display
import ipywidgets as w

from jobspy import scrape_jobs

jobs = None

# --- Site names (populates list) ---
SITE_OPTIONS = [
    ("Indeed", "indeed"),
    ("LinkedIn", "linkedin"),
    ("ZipRecruiter", "zip_recruiter"),
    ("Google", "google"),
    ("Glassdoor", "glassdoor"),
    ("Bayt", "bayt"),
    ("Naukri", "naukri"),
    ("BDJobs", "bdjobs"),
]
site_names = w.SelectMultiple(
    options=[(label, val) for label, val in SITE_OPTIONS],
    value=["indeed"],
    description="Sites",
    rows=5,
    layout=w.Layout(width="320px"),
)

# --- Search term (text) ---
search_term = w.Text(
    value="software engineer",
    description="Search",
    style={"description_width": "80px"},
    layout=w.Layout(width="520px"),
)
google_search_term = w.Text(
    value="",
    placeholder="For Google only: e.g. software engineer jobs near San Francisco, CA since yesterday",
    description="Google q",
    style={"description_width": "80px"},
    layout=w.Layout(width="520px"),
)

# --- Location: dropdown + Other ---
LOCATION_PRESETS = [
    "San Francisco, CA",
    "New York, NY",
    "Remote",
    "Chicago, IL",
    "Boston, MA",
    "Seattle, WA",
    "Austin, TX",
    "London, UK",
    "Toronto, ON",
    "Other",
]
location_dropdown = w.Dropdown(options=LOCATION_PRESETS, value="San Francisco, CA", description="Location", layout=w.Layout(width="260px"))
location_other = w.Text(placeholder="Custom location when 'Other' selected", description="Custom", layout=w.Layout(width="340px"))

def on_location_change(change):
    location_other.layout.display = "block" if change["new"] == "Other" else "none"

location_dropdown.observe(on_location_change, names="value")
location_other.layout.display = "none"

# --- Remote & other params ---
is_remote = w.Checkbox(value=False, description="Remote only")
results_wanted = w.BoundedIntText(value=20, min=1, max=300, step=1, description="Results", layout=w.Layout(width="180px"))
hours_old_options = [(f"{h} days", h * 24) for h in [1, 3, 7, 14, 30]]
hours_old = w.Dropdown(options=hours_old_options, value=72, description="Posted", layout=w.Layout(width="180px"))

# --- Country (Indeed/Glassdoor) ---
COUNTRIES = [
    "USA", "UK", "Canada", "India", "Australia", "Germany", "France", "Netherlands",
    "Spain", "Italy", "Ireland", "Singapore", "Brazil", "Mexico", "South Africa",
    "United Arab Emirates", "New Zealand", "Belgium", "Austria", "Switzerland",
    "Japan", "South Korea", "Indonesia", "Malaysia", "Philippines", "Vietnam",
    "Argentina", "Chile", "Colombia", "Poland", "Sweden", "Portugal", "Israel",
]
country_indeed = w.Dropdown(options=COUNTRIES, value="USA", description="Country", layout=w.Layout(width="210px"))

# --- Job type and controls ---
job_type = w.Dropdown(
    options=[("(any)", None), ("Full-time", "fulltime"), ("Part-time", "parttime"), ("Internship", "internship"), ("Contract", "contract")],
    value=None,
    description="Type",
    layout=w.Layout(width="220px"),
)
distance = w.BoundedIntText(value=50, min=1, max=200, step=1, description="Distance", layout=w.Layout(width="190px"))
description_format = w.Dropdown(options=[("Markdown", "markdown"), ("HTML", "html")], value="markdown", description="Desc", layout=w.Layout(width="190px"))
verbose = w.Dropdown(options=[("Quiet", 0), ("Warnings", 1), ("All logs", 2)], value=1, description="Logs", layout=w.Layout(width="170px"))
easy_apply = w.Checkbox(value=False, description="Easy apply")

status = w.HTML("<span style='color:#475569;font-weight:500'>Ready to run</span>")

# --- Output and Run ---
out = w.Output(layout=w.Layout(border="1px solid #cbd5e1", padding="10px", border_radius="10px", background_color="#ffffff"))

def run_search(_):
    global jobs
    with out:
        out.clear_output()

        loc = location_other.value.strip() if location_dropdown.value == "Other" else location_dropdown.value
        if location_dropdown.value == "Other" and not loc:
            status.value = "<span style='color:#b00;font-weight:600'>Please enter a custom location.</span>"
            print("Please enter a custom location.")
            return

        selected_sites = list(site_names.value) or ["indeed"]
        kwargs = {
            "site_name": selected_sites,
            "search_term": search_term.value.strip() or "software engineer",
            "location": loc,
            "results_wanted": int(results_wanted.value),
            "country_indeed": country_indeed.value,
            "verbose": int(verbose.value),
            "description_format": description_format.value,
            "distance": int(distance.value),
        }

        # Respect known library limitations to avoid invalid parameter combinations.
        has_indeed = "indeed" in selected_sites
        has_linkedin = "linkedin" in selected_sites

        # Keep hours_old by default; suppress conflicting filters when needed.
        kwargs["hours_old"] = int(hours_old.value)

        if easy_apply.value and has_linkedin and has_indeed:
            status.value = "<span style='color:#b80;font-weight:600'>Using hours_old only due to LinkedIn+Indeed limitations.</span>"
        elif easy_apply.value and has_linkedin:
            kwargs["easy_apply"] = True
        elif easy_apply.value and has_indeed:
            kwargs.pop("hours_old", None)
            kwargs["easy_apply"] = True

        if job_type.value is not None:
            # Indeed cannot combine job_type/is_remote with hours_old.
            if has_indeed and "hours_old" in kwargs:
                kwargs.pop("hours_old", None)
            kwargs["job_type"] = job_type.value

        if is_remote.value:
            if has_indeed and "hours_old" in kwargs:
                kwargs.pop("hours_old", None)
            kwargs["is_remote"] = True

        if google_search_term.value.strip():
            kwargs["google_search_term"] = google_search_term.value.strip()

        status.value = "<span style='color:#0369a1;font-weight:600'>Running search...</span>"
        print("Running scrape_jobs with:", kwargs)

        try:
            jobs = scrape_jobs(**kwargs)
        except Exception as e:
            jobs = None
            status.value = f"<span style='color:#b00;font-weight:600'>Failed: {e}</span>"
            print("Error:", e)
            return

        count = len(jobs) if jobs is not None else 0
        print(f"Found {count} jobs")

        if jobs is not None and not jobs.empty:
            cols = [c for c in ["site", "title", "company", "location", "job_url"] if c in jobs.columns]
            ipy_display(jobs[cols].head(25) if cols else jobs.head(25))
            jobs.to_csv("jobs.csv", quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
            print("Saved to jobs.csv")
            status.value = f"<span style='color:#15803d;font-weight:600'>Done. Found {count} jobs.</span>"
        else:
            status.value = "<span style='color:#b45309;font-weight:600'>No jobs returned.</span>"
            print("No jobs returned.")

run_btn = w.Button(
    description="Run search",
    button_style="primary",
    icon="search",
    layout=w.Layout(width="160px", height="38px"),
)
run_btn.on_click(run_search)

# Layout (portfolio-friendly card style)
header = w.HTML(
    """
    <div style='padding:10px 12px;border:1px solid #e2e8f0;border-radius:10px;background:#f8fafc;'>
        <h4 style='margin:0 0 4px 0;color:#0f172a'>JobSpy Search Console</h4>
        <p style='margin:0;color:#475569;font-size:13px'>Tune filters, run fast tests, and export curated results for portfolio demos.</p>
    </div>
    """
)
left = w.VBox(
    [w.HTML("<b>Boards</b><div style='font-size:12px;color:#64748b'>Pick one or more sources</div>"), site_names],
    layout=w.Layout(width="340px", gap="8px"),
)
right = w.VBox(
    [
        search_term,
        google_search_term,
        w.HBox([location_dropdown, location_other]),
        w.HBox([results_wanted, hours_old, country_indeed]),
        w.HBox([job_type, distance, description_format, verbose]),
        w.HBox([is_remote, easy_apply]),
        w.HBox([run_btn, status], layout=w.Layout(align_items="center", gap="10px")),
    ],
    layout=w.Layout(gap="8px"),
)

form = w.VBox(
    [
        header,
        w.HBox([left, right], layout=w.Layout(gap="14px", align_items="flex-start")),
        out,
    ],
    layout=w.Layout(gap="10px"),
)
ipy_display(form)

## 3. Inspect results

In [ ]:
if "jobs" not in globals() or jobs is None:
    print("No jobs in memory yet. Run the Search UI cell first.")
elif jobs.empty:
    print("jobs exists but is empty.")
else:
    print("Shape:", jobs.shape)
    print("Columns:", list(jobs.columns))
    cols = [c for c in ["site", "title", "company", "location", "job_url"] if c in jobs.columns]
    try:
        from IPython.display import display
        display(jobs[cols].head() if cols else jobs.head())
    except Exception:
        print((jobs[cols].head() if cols else jobs.head()).to_string())

## 4. Optional: export to CSV

In [ ]:
import csv

out_path = "jobs.csv"
if "jobs" not in globals() or jobs is None:
    print("Nothing to save. Run the Search UI cell first.")
elif jobs.empty:
    print("jobs exists but is empty. Nothing to save.")
else:
    jobs.to_csv(out_path, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
    print(f"Saved to {out_path}")
    # For Marimo: after export you can free memory with: del jobs

## 5. Multi-site test (optional, more requests)

In [ ]:
# Uncomment to try multiple sites. LinkedIn rate-limits quickly; use proxies for heavy use.
# jobs_multi = scrape_jobs(
#     site_name=["indeed", "zip_recruiter"],
#     search_term="data analyst",
#     location="Remote",
#     results_wanted=5,
#     country_indeed="USA",
#     is_remote=True,
#     verbose=1,
# )
# print(jobs_multi[["site", "title", "company"]].head() if jobs_multi is not None else "No results")